# ️ Oil & Gas Wells & Glossary RAG System Pipeline
### End-to-End Notebook: Ingestion, Parsing, Chunking, Hybrid Vector Indexing, and Guardrailed QA



### 1. Setup Environment & Konfigurasi


In [9]:
import os
import re
import json
import sqlite3
import zipfile
import xml.etree.ElementTree as ET
from pathlib import Path
from typing import List, Dict, Any

import pypdf
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv

# Load environment variables dari root project
load_dotenv(dotenv_path='../.env')

# Setup OpenAI Client
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY tidak ditemukan di .env")
client = OpenAI(api_key=OPENAI_API_KEY)

# Konfigurasi Path Dokumen
BASE_DIR = Path("..").resolve()
DATASETS_DIR = BASE_DIR / "Datasets"
GLOSSARY_PATH = BASE_DIR / "Glossaries.docx"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

print("Base Directory:", BASE_DIR)
print("Datasets Directory:", DATASETS_DIR, "Exists:", DATASETS_DIR.exists())
print("Glossary Path:", GLOSSARY_PATH, "Exists:", GLOSSARY_PATH.exists())


Base Directory: C:\Users\FS-User\Documents\prj\wells-info
Datasets Directory: C:\Users\FS-User\Documents\prj\wells-info\Datasets Exists: True
Glossary Path: C:\Users\FS-User\Documents\prj\wells-info\Glossaries.docx Exists: True


### 2. Pemrosesan Data (Parsing)
#### 2.1 Parsing Dokumen Glosarium (`Glossaries.docx`)
Mengekstrak singkatan, istilah, dan definisi dari file Word docx.


In [10]:
def parse_docx_tables_and_text(docx_path: Path):
    """Ekstrak teks dan tabel dari file .docx secara terstruktur."""
    with zipfile.ZipFile(docx_path) as docx:
        tree = ET.fromstring(docx.read('word/document.xml'))
        ns = {'w': 'http://schemas.openxmlformats.org/wordprocessingml/2006/main'}
        
        paragraphs = []
        for p in tree.iterfind('.//w:p', ns):
            p_text = ''.join(node.text for node in p.iterfind('.//w:t', ns) if node.text)
            if p_text.strip():
                paragraphs.append(p_text.strip())
                
    glossary_items = []
    
    for i, line in enumerate(paragraphs):
        if " - " in line or ":" in line:
            parts = re.split(r'[\-\:]', line, 1)
            term = parts[0].strip()
            meaning = parts[1].strip() if len(parts) > 1 else ""
            glossary_items.append({"term": term, "meaning": meaning, "raw": line})
        elif i + 1 < len(paragraphs) and len(line) < 30 and len(paragraphs[i+1]) > 20:
            glossary_items.append({"term": line, "meaning": paragraphs[i+1], "raw": f"{line}: {paragraphs[i+1]}"})
            
    chunks = []
    # 1. Chunk full text
    full_text = "\n".join(paragraphs)
    chunks.append({
        "id": "glossary_full",
        "source_file": docx_path.name,
        "doc_type": "Glossary",
        "category": "Oil and Gas Terms",
        "title": "Glosarium Singkatan dan Istilah Lengkap",
        "content": full_text,
        "metadata": {"total_items": len(glossary_items)}
    })
    
    # 2. Chunk per kelompok istilah
    chunk_size = 5
    for idx in range(0, len(glossary_items), chunk_size):
        batch = glossary_items[idx:idx+chunk_size]
        batch_text = "\n".join([f"- {item['term']}: {item['meaning']}" for item in batch if item['term']])
        terms_list = [item['term'] for item in batch if item['term']]
        chunks.append({
            "id": f"glossary_chunk_{idx//chunk_size + 1}",
            "source_file": docx_path.name,
            "doc_type": "Glossary",
            "category": "Oil and Gas Terms",
            "title": f"Glosarium Istilah Part {idx//chunk_size + 1}",
            "content": batch_text,
            "metadata": {"terms": terms_list}
        })
        
    return chunks, glossary_items

glossary_chunks, glossary_items = parse_docx_tables_and_text(GLOSSARY_PATH)
print(f"Extracted {len(glossary_items)} glossary items into {len(glossary_chunks)} search chunks.")
print("Contoh chunk glossary:\n", glossary_chunks[1]['content'][:250])


Extracted 173 glossary items into 36 search chunks.
Contoh chunk glossary:
 - BARAKUDA: Field name. The well targets reservoirs below the Tapis field.
- AFE: Authorization for Expenditure – Approved budget and plan for the well.
- AMSL: Above Mean Sea Level – Height above average sea level.
- API: American Petroleum Institut


#### 2.2 Parsing Dokumen PDF Sumur (`Datasets/`)
Mendukung laporan **DGOS** (Daily Geological Operations Summary) dan **DOR/DDR** (Daily Operation Report). Parsing mengekstrak metadata spesifik seperti Lokasi (Region/Block), NPT, Rig Name, Wireline runs, Operasi harian, dan Mud Weight.


In [11]:
def extract_pdf_metadata_and_sections(pdf_path: Path):
    """Ekstrak teks PDF dan mengidentifikasi bagian-bagian penting serta metadata."""
    reader = pypdf.PdfReader(str(pdf_path))
    total_pages = len(reader.pages)
    
    extracted_chunks = []
    full_pdf_text = ""
    
    for page_idx, page in enumerate(reader.pages):
        page_text = page.extract_text() or ""
        full_pdf_text += f"\n--- Page {page_idx + 1} ---\n" + page_text
        
        cleaned_text = re.sub(r' +', ' ', page_text).strip()
        
        extracted_chunks.append({
            "id": f"{pdf_path.stem}_p{page_idx+1}",
            "source_file": pdf_path.name,
            "doc_type": "DGOS" if "DGOS" in pdf_path.name else "DDR",
            "page": page_idx + 1,
            "total_pages": total_pages,
            "content": cleaned_text,
            "metadata": {
                "source_file": pdf_path.name,
                "page": page_idx + 1
            }
        })
        
    summary_metadata = {}
    
    # Deteksi Nama Sumur
    if "BARAKUDA-1" in full_pdf_text or "BARAKUDA" in pdf_path.name:
        summary_metadata["well_name"] = "BARAKUDA-1"
    
    # Deteksi Region & Negara
    region_match = re.search(r'Region\s*:\s*([A-Za-z0-9\-\_]+)', full_pdf_text)
    if region_match:
        summary_metadata["region"] = region_match.group(1).strip()
    if "PM" in full_pdf_text or "PM515" in full_pdf_text:
        summary_metadata["region"] = "PM (Peninsular Malaysia)"
        summary_metadata["country"] = "Malaysia"
        
    block_match = re.search(r'Block\s*:\s*([A-Za-z0-9\-\_]+)', full_pdf_text)
    if block_match:
        summary_metadata["block"] = block_match.group(1).strip()
        
    # Deteksi NPT
    npt_matches = re.findall(r'NPT[\:\s]+([^\n\r]+)', full_pdf_text, re.IGNORECASE)
    if npt_matches:
        summary_metadata["npt_info"] = [m.strip() for m in npt_matches if m.strip()]
        
    # Deteksi Wireline / Next Update
    wl_matches = re.findall(r'(WL Run[^\.\n\r]+|Wireline[^\.\n\r]+|Perform WL[^\.\n\r]+)', full_pdf_text, re.IGNORECASE)
    if wl_matches:
        summary_metadata["wireline_info"] = [w.strip() for w in wl_matches]
        
    # Tambahkan document summary chunk
    extracted_chunks.insert(0, {
        "id": f"{pdf_path.stem}_summary",
        "source_file": pdf_path.name,
        "doc_type": "Document Summary",
        "page": 0,
        "total_pages": total_pages,
        "content": (
            f"Dokumen: {pdf_path.name}\n"
            f"Well Name: {summary_metadata.get('well_name', 'BARAKUDA-1')}\n"
            f"Lokasi / Region / Country: {summary_metadata.get('country', 'Malaysia')} ({summary_metadata.get('region', 'PM Block PM515')})\n"
            f"Ringkasan Isi & Metadata:\n{full_pdf_text[:1200]}"
        ),
        "metadata": summary_metadata
    })
    
    return extracted_chunks

# Parse semua PDF di Datasets
all_chunks = []
all_chunks.extend(glossary_chunks)

for root, dirs, files in os.walk(DATASETS_DIR):
    for f in sorted(files):
        if f.endswith(".pdf"):
            pdf_path = Path(root) / f
            pdf_chunks = extract_pdf_metadata_and_sections(pdf_path)
            all_chunks.extend(pdf_chunks)
            print(f"Parsed '{f}': {len(pdf_chunks)} chunks")

print(f"\nTotal parsed chunks siap index: {len(all_chunks)}")


Parsed 'BARAKUDA-1_DGOS_72_20260829.pdf': 2 chunks
Parsed 'BARAKUDA-1_DGOS_84_20260910.pdf': 2 chunks
Parsed 'NAGA-2_BARAKUDA-1_DDR_32_19_07_2026_Drill_17_5in_x_20in_Hole.pdf': 8 chunks
Parsed 'NAGA-2_BARAKUDA-1_DDR_53_09_08_2026_LD_14_75in_Motor_BHA__Wireline_Operation.pdf': 7 chunks

Total parsed chunks siap index: 55


#### 2.3 Simpan Data Terstruktur ke JSON & SQLite
Menyimpan hasil parsing ke file JSON dan database SQLite (`wells_data.db`).


In [12]:
# 1. Simpan ke JSON
json_output_path = OUTPUT_DIR / "parsed_wells_data.json"
with open(json_output_path, "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, indent=2, ensure_ascii=False)

print(f"JSON hasil parsing disimpan di: {json_output_path} ({os.path.getsize(json_output_path)} bytes)")

# 2. Simpan ke Database SQLite (Nice-to-have requirement)
db_path = OUTPUT_DIR / "wells_data.db"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS parsed_documents (
    id TEXT PRIMARY KEY,
    source_file TEXT,
    doc_type TEXT,
    page INTEGER,
    content TEXT,
    metadata_json TEXT
)
""")

cursor.execute("""
CREATE VIRTUAL TABLE IF NOT EXISTS documents_fts USING fts5(
    id,
    source_file,
    content
)
""")

cursor.execute("DELETE FROM parsed_documents")
cursor.execute("DELETE FROM documents_fts")

for chunk in all_chunks:
    cursor.execute("""
        INSERT INTO parsed_documents (id, source_file, doc_type, page, content, metadata_json)
        VALUES (?, ?, ?, ?, ?, ?)
    """, (
        chunk["id"],
        chunk["source_file"],
        chunk.get("doc_type", "General"),
        chunk.get("page", 0),
        chunk["content"],
        json.dumps(chunk.get("metadata", {}))
    ))
    cursor.execute("""
        INSERT INTO documents_fts (id, source_file, content)
        VALUES (?, ?, ?)
    """, (chunk["id"], chunk["source_file"], chunk["content"]))

conn.commit()
conn.close()
print(f"Database SQLite berhasil dibuat: {db_path}")


JSON hasil parsing disimpan di: C:\Users\FS-User\Documents\prj\wells-info\output\parsed_wells_data.json (94541 bytes)
Database SQLite berhasil dibuat: C:\Users\FS-User\Documents\prj\wells-info\output\wells_data.db


### 3. Embedding Generation & Hybrid Index
Menghasilkan vector embeddings untuk setiap chunk menggunakan OpenAI `text-embedding-3-small` dan menyimpan cache embedding secara lokal.


In [13]:
def get_embeddings(texts: List[str], model: str = "text-embedding-3-small") -> List[List[float]]:
    """Menghasilkan embedding vektor menggunakan OpenAI API."""
    cleaned_texts = [t.replace("\n", " ")[:8000] for t in texts]
    response = client.embeddings.create(input=cleaned_texts, model=model)
    return [item.embedding for item in response.data]

# Embed semua chunk
print(f"Generating embeddings for {len(all_chunks)} chunks...")
chunk_texts = [f"Dokumen: {c['source_file']} | Konten: {c['content']}" for c in all_chunks]

embeddings_matrix = get_embeddings(chunk_texts)
print(f"Embeddings shape: {len(embeddings_matrix)} x {len(embeddings_matrix[0])}")

# Simpan embeddings ke dalam file lokal untuk caching cepat
embedding_cache_path = OUTPUT_DIR / "embeddings_cache.npz"
np.savez_compressed(embedding_cache_path, embeddings=np.array(embeddings_matrix))
print(f"Embeddings cache tersimpan di: {embedding_cache_path}")


Generating embeddings for 55 chunks...
Embeddings shape: 55 x 1536
Embeddings cache tersimpan di: C:\Users\FS-User\Documents\prj\wells-info\output\embeddings_cache.npz


### 4. Hybrid Search Engine (Vector Cosine Similarity + Keyword Filter)
Menggabungkan pencarian semantik (vektor) dengan pencarian kata kunci (exact match & keyword overlap) agar akurat pada istilah spesifik seperti kode alat, singkatan, angka NPT, dan nama sumur.


In [14]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Hitung cosine similarity antara query vector dan embedding matrix."""
    return np.dot(b, a) / (np.linalg.norm(b, axis=1) * np.linalg.norm(a) + 1e-10)

def retrieve_relevant_chunks(query: str, top_k: int = 5) -> List[Dict[str, Any]]:
    """Hybrid retrieval: Kombinasi Semantic Vector Search & Keyword Matching."""
    # 1. Vector Search
    query_vec = np.array(get_embeddings([query])[0])
    emb_matrix = np.array(embeddings_matrix)
    vector_scores = cosine_similarity(query_vec, emb_matrix)
    
    # 2. Keyword Boost
    query_keywords = re.findall(r'\w+', query.lower())
    keyword_scores = np.zeros(len(all_chunks))
    
    for idx, chunk in enumerate(all_chunks):
        content_lower = chunk["content"].lower()
        file_lower = chunk["source_file"].lower()
        matches = sum(1 for kw in query_keywords if kw in content_lower or kw in file_lower)
        keyword_scores[idx] = matches / (len(query_keywords) + 1)
        
    # Hybrid Score: 70% vector + 30% keyword
    final_scores = 0.7 * vector_scores + 0.3 * keyword_scores
    top_indices = np.argsort(final_scores)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        results.append({
            "chunk": all_chunks[idx],
            "score": float(final_scores[idx]),
            "vector_score": float(vector_scores[idx]),
            "keyword_score": float(keyword_scores[idx])
        })
    return results

# Uji retrieval
test_retrieval = retrieve_relevant_chunks("Wireline run apa yang direncanakan ?", top_k=3)
for r in test_retrieval:
    print(f"[Score: {r['score']:.3f}] Source: {r['chunk']['source_file']} | Sample: {r['chunk']['content'][:120]}...")


[Score: 0.441] Source: NAGA-2_BARAKUDA-1_DDR_53_09_08_2026_LD_14_75in_Motor_BHA__Wireline_Operation.pdf | Sample: Daily Operation Report
Wellbore No.: OH Report no.: 53 Report date: 09/08/2026
OPERATION SUMMARY
From - To Hrs Phase 
Co...
[Score: 0.411] Source: NAGA-2_BARAKUDA-1_DDR_53_09_08_2026_LD_14_75in_Motor_BHA__Wireline_Operation.pdf | Sample: Daily Operation Report
Wellbore No.: OH Report no.: 53 Report date: 09/08/2026
OPERATION SUMMARY
From - To Hrs Phase 
Co...
[Score: 0.399] Source: NAGA-2_BARAKUDA-1_DDR_53_09_08_2026_LD_14_75in_Motor_BHA__Wireline_Operation.pdf | Sample: Dokumen: NAGA-2_BARAKUDA-1_DDR_53_09_08_2026_LD_14_75in_Motor_BHA__Wireline_Operation.pdf
Well Name: BARAKUDA-1
Lokasi /...


### 5. RAG Response Generation & Guardrails


In [15]:
SYSTEM_PROMPT = """Anda adalah AI Asisten Ahli Data Operasional Sumur & Glosarium Oil & Gas (Wells & Petroleum AI Specialist).

TUGAS UTAMA:
1. Jawab pertanyaan pengguna HANYA berdasarkan KELENGKAPAN INFORMASI dari KONTEKS DOKUMEN yang diberikan.
2. WAJIB menyertakan NAMA FILE SUMBER yang relevan pada akhir jawaban dalam format:
   `[Sumber: <nama_file.pdf atau Glossaries.docx>]`
3. Jawab dengan ringkas, padat, faktual, dan sangat akurat sesuai data.

ATURAN GUARDRAIL (PENOLAKAN DI LUAR CAKUPAN):
- Jika pertanyaan pengguna di luar lingkup informasi sumur, operasi migas, atau istilah glosarium yang ada dalam dokumen (misal: topik umum, hiburan, resep masakan, dll.), TOLAK DENGAN PESAN KONSISTEN:
  "Maaf, pertanyaan Anda berada di luar cakupan data yang tersedia. Saya hanya dapat menjawab pertanyaan seputar operasional sumur (BARAKUDA-1, DGOS, DDR/DOR) dan glosarium istilah Oil & Gas yang tersedia di dataset."
- Jangan pernah mengarang data (no hallucination). Jika informasi tidak ada di dokumen, sampaikan bahwa informasi tidak ditemukan dalam dataset.
"""

def answer_question(query: str, model: str = "gpt-4o-mini") -> Dict[str, Any]:
    """Menjawab pertanyaan menggunakan Hybrid RAG dengan Guardrails."""
    # Ambil konteks relevan
    retrieval_results = retrieve_relevant_chunks(query, top_k=5)
    
    # Format context
    context_str = ""
    sources_cited = set()
    for idx, item in enumerate(retrieval_results):
        ch = item["chunk"]
        sources_cited.add(ch["source_file"])
        context_str += f"\n--- DOKUMEN {idx+1}: {ch['source_file']} (Halaman: {ch.get('page', 'N/A')}) ---\n{ch['content']}\n"
        
    user_prompt = f"""Konteks Dokumen yang Ditemukan:
{context_str}

Pertanyaan Pengguna: {query}
Berikan jawaban terbaik berdasarkan konteks di atas:"""

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.0
    )
    
    answer_text = response.choices[0].message.content
    return {
        "query": query,
        "answer": answer_text,
        "sources": list(sources_cited),
        "retrieved_chunks": retrieval_results
    }


### 6. Evaluasi & Pengujian Benchmark Pertanyaan Uji



In [16]:
test_queries = [
    # Benchmark dari Instruction.docx
    "Dimana letak lokasi sumur ?",
    "Berapa Total NPT sumur ?",
    "Wireline run apa yang direncanakan ?",
    
    # Pengujian Glosarium
    "Apa arti dari istilah BHA, NPT, dan MDT pada glosarium migas?",
    
    # Pengujian Guardrail Out-of-Scope
    "Siapa presiden Amerika Serikat saat ini dan bagaimana cuaca di Jakarta hari ini?"
]

print("=" * 80)
print("RUNNING RAG BENCHMARK TESTS")
print("=" * 80)

for i, q in enumerate(test_queries, 1):
    result = answer_question(q)
    print(f"\n\n[TEST {i}] Pertanyaan: {q}")
    print("-" * 80)
    print("Jawaban AI:")
    print(result["answer"])
    print("-" * 80)


RUNNING RAG BENCHMARK TESTS


[TEST 1] Pertanyaan: Dimana letak lokasi sumur ?
--------------------------------------------------------------------------------
Jawaban AI:
Lokasi sumur BARAKUDA-1 terletak di Malaysia, tepatnya di Peninsular Malaysia, dengan koordinat 05° 30' 29.345'' N (latitude) dan 105° 01' 28.448'' E (longitude). [Sumber: BARAKUDA-1_DGOS_72_20260829.pdf]
--------------------------------------------------------------------------------


[TEST 2] Pertanyaan: Berapa Total NPT sumur ?
--------------------------------------------------------------------------------
Jawaban AI:
Total Non-Productive Time (NPT) untuk sumur BARAKUDA-1 adalah 1.50 jam. [Sumber: NAGA-2_BARAKUDA-1_DDR_32_19_07_2026_Drill_17_5in_x_20in_Hole.pdf]
--------------------------------------------------------------------------------


[TEST 3] Pertanyaan: Wireline run apa yang direncanakan ?
--------------------------------------------------------------------------------
Jawaban AI:
Wireline run yang di